# 04 — Integración

Fase final de **Data Preparation** de CRISP-DM. Unimos los tres datasets limpios en un único fichero `dataset_integrado.csv` con granularidad **listing × día**, listo para:

- Subir a BigQuery como `datos.fact_ocupacion`.
- Importar en Orange Data Mining para entrenar los clasificadores.
- Conectar desde Power BI para el dashboard final.

> `eventos_clean.csv` no se integra en el modelo (limitación de Ticketmaster, documentada en el notebook 03) pero sí se sube a BigQuery como tabla auxiliar.

## 0. Imports y carga de ficheros limpios

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

PROC_DIR = Path('../data/processed')

listings  = pd.read_csv(PROC_DIR / 'listings_clean.csv')
calendar  = pd.read_csv(PROC_DIR / 'calendar_clean.csv', parse_dates=['date'])
clima     = pd.read_csv(PROC_DIR / 'clima_clean.csv',   parse_dates=['date'])

print(f'Listings : {listings.shape}')
print(f'Calendar : {calendar.shape}')
print(f'Clima    : {clima.shape}')

---
## 1. Join calendar ⟕ listings

La tabla base es `calendar` (1 fila por listing × día). Añadimos las características estáticas de `listings` por `listing_id`.

In [ ]:
# Renombrar id para poder hacer merge
listings_j = listings.rename(columns={'id': 'listing_id'})

df = calendar.merge(listings_j, on='listing_id', how='left', validate='many_to_one')
print(f'Tras join con listings: {df.shape}')

# Comprobar que no hemos perdido filas ni generado duplicados
assert len(df) == len(calendar), 'El join listings añadió o perdió filas'
huerfanos = df['neighbourhood_cleansed'].isna().sum()
print(f'Filas sin match de listing: {huerfanos}')

---
## 2. Join con clima

In [ ]:
df = df.merge(clima, on='date', how='left', validate='many_to_one')
print(f'Tras join con clima: {df.shape}')

huerf_clima = df['temperature_2m_max'].isna().sum()
print(f'Filas sin datos de clima: {huerf_clima}')

# Si quedan huérfanos (fechas sin clima), imputar con la media global
if huerf_clima > 0:
    for col in ['temperature_2m_max','temperature_2m_min','precipitation_sum','windspeed_10m_max','temp_avg']:
        df[col] = df[col].fillna(df[col].mean())
    df['weather_cat'] = df['weather_cat'].fillna('nublado')
    df['has_rain']    = df['has_rain'].fillna(0).astype(int)
    df['weathercode'] = df['weathercode'].fillna(3)
    print('Nulos de clima imputados con la media del horizonte')

---
## 3. Limpieza final y reorden

In [ ]:
# Quitar columnas redundantes o que ya están codificadas en otras
drop_cols = ['weathercode', 'fuente',            # weather_cat ya resume weathercode; fuente es metadato
             'minimum_nights_y', 'maximum_nights_y']   # los del listing prevalecen, no los del día concreto
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

# Renombrar por si han quedado sufijos _x / _y tras merges
df = df.rename(columns={
    'minimum_nights_x': 'minimum_nights_day',
    'maximum_nights_x': 'maximum_nights_day',
})

# Reordenar: identificadores → objetivo → features
orden = [
    'listing_id', 'date',
    'occupied',   # objetivo binario
    # Temporales
    'dow', 'is_weekend', 'month', 'day', 'week', 'is_holiday',
    # Listing
    'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'latitude', 'longitude',
    'property_type', 'room_type',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'estimated_occupancy_l365d',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_location', 'review_scores_value',
    'host_is_superhost', 'instant_bookable',
    # Clima
    'temperature_2m_max', 'temperature_2m_min', 'temp_avg',
    'precipitation_sum', 'windspeed_10m_max',
    'weather_cat', 'has_rain',
]
orden = [c for c in orden if c in df.columns]
resto = [c for c in df.columns if c not in orden]
df = df[orden + resto]
print(f'Dataset final: {df.shape}')
print(f'Columnas: {list(df.columns)}')

---
## 4. Validación final

In [ ]:
print('=== Distribución de la variable objetivo ===')
print(df['occupied'].value_counts())
print(f'% ocupado: {df["occupied"].mean()*100:.2f}%')

print('\n=== Nulos por columna (solo > 0) ===')
n = df.isna().sum()
n = n[n > 0]
print(n if len(n) else 'Sin nulos')

print('\n=== Rango de fechas ===')
print(f'{df["date"].min().date()} → {df["date"].max().date()}  ({df["date"].nunique()} días)')
print(f'Listings únicos: {df["listing_id"].nunique():,}')
print(f'Filas totales  : {len(df):,}')

---
## 5. Guardar

In [ ]:
output_path = PROC_DIR / 'dataset_integrado.csv'
df.to_csv(output_path, index=False)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f'Guardado: {output_path}')
print(f'Tamaño: {size_mb:.1f} MB')
print(f'Shape: {df.shape}')
df.head()

---
## 6. Siguiente paso — notebook 05 (modelado)

`dataset_integrado.csv` está listo para:

1. **BigQuery:** subir como `datos.fact_ocupacion` (ver notebook 06 o script aparte).
2. **Orange Data Mining:** arrastrar el CSV al canvas, seleccionar `occupied` como target y lanzar Logistic Regression / Random Forest / Gradient Boosting.
3. **Power BI:** conectar al proyecto de BigQuery y montar el dashboard con KPIs por barrio, room_type y clima.

En el **notebook 05** haremos el modelado en Python (sklearn) como *baseline* antes del trabajo visual con Orange:
- Split train/test por fecha (no aleatorio, para simular predicción).
- Modelos: LogisticRegression, RandomForestClassifier, GradientBoostingClassifier.
- Métricas: Accuracy, F1, AUC, matriz de confusión.
- Importancia de variables.